In [1]:
# ============================================================
# SUPPLY CHAIN INTELLIGENCE SYSTEM
# Notebook 2: Data Cleaning & Validation
# ============================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Libraries imported!")

✅ Libraries imported!


In [2]:
# ============================================================
# Load all 5 raw tables
# ============================================================

path = "D:/Projects/End-to-end projects/8. Supply Chain Intelligence/Data/Raw"

df_products  = pd.read_csv(path + "/products.csv")
df_suppliers = pd.read_csv(path + "/suppliers.csv")
df_po        = pd.read_csv(path + "/purchase_orders.csv")
df_inventory = pd.read_csv(path + "/inventory_snapshots.csv")
df_sales     = pd.read_csv(path + "/sales_orders.csv")

print("✅ All tables loaded!")
print(f"\n{'Table':<25} {'Rows':>8} {'Columns':>10}")
print("-" * 45)
print(f"{'Products':<25} {len(df_products):>8,} {len(df_products.columns):>10}")
print(f"{'Suppliers':<25} {len(df_suppliers):>8,} {len(df_suppliers.columns):>10}")
print(f"{'Purchase Orders':<25} {len(df_po):>8,} {len(df_po.columns):>10}")
print(f"{'Inventory Snapshots':<25} {len(df_inventory):>8,} {len(df_inventory.columns):>10}")
print(f"{'Sales Orders':<25} {len(df_sales):>8,} {len(df_sales.columns):>10}")

✅ All tables loaded!

Table                         Rows    Columns
---------------------------------------------
Products                        50         10
Suppliers                       15         12
Purchase Orders              1,960         13
Inventory Snapshots         15,750         10
Sales Orders                 2,600         16


In [3]:
# ============================================================
# Data Quality Check — All Tables
# ============================================================

tables = {
    'Products':            df_products,
    'Suppliers':           df_suppliers,
    'Purchase Orders':     df_po,
    'Inventory Snapshots': df_inventory,
    'Sales Orders':        df_sales
}

for name, df in tables.items():
    print(f"\n{'='*50}")
    print(f"TABLE: {name}")
    print(f"{'='*50}")
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    
    # Missing values
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f"\n⚠️  Missing Values Found:")
        for col, count in missing.items():
            pct = count / len(df) * 100
            print(f"   {col:<35} {count:>6,} ({pct:.1f}%)")
    else:
        print(f"\n✅ No missing values")
    
    # Duplicates
    dupes = df.duplicated().sum()
    if dupes > 0:
        print(f"\n⚠️  Duplicate rows: {dupes:,}")
    else:
        print(f"✅ No duplicate rows")


TABLE: Products
Shape: 50 rows × 10 columns

✅ No missing values
✅ No duplicate rows

TABLE: Suppliers
Shape: 15 rows × 12 columns

✅ No missing values
✅ No duplicate rows

TABLE: Purchase Orders
Shape: 1,960 rows × 13 columns

⚠️  Missing Values Found:
   actual_delivery_date                    41 (2.1%)
   delay_days                              41 (2.1%)
✅ No duplicate rows

TABLE: Inventory Snapshots
Shape: 15,750 rows × 10 columns

✅ No missing values
✅ No duplicate rows

TABLE: Sales Orders
Shape: 2,600 rows × 16 columns

✅ No missing values
✅ No duplicate rows


In [4]:
# ============================================================
# Fix Data Types — Convert dates, ensure numerics are correct
# ============================================================

# --- Products ---
df_products['unit_cost_inr']      = pd.to_numeric(df_products['unit_cost_inr'])
df_products['selling_price_inr']  = pd.to_numeric(df_products['selling_price_inr'])
df_products['gross_margin_pct']   = pd.to_numeric(df_products['gross_margin_pct'])

# --- Suppliers ---
df_suppliers['contract_start_date'] = pd.to_datetime(df_suppliers['contract_start_date'])
df_suppliers['avg_lead_time_days']  = pd.to_numeric(df_suppliers['avg_lead_time_days'])

# --- Purchase Orders ---
df_po['order_date']             = pd.to_datetime(df_po['order_date'])
df_po['promised_delivery_date'] = pd.to_datetime(df_po['promised_delivery_date'])
df_po['actual_delivery_date']   = pd.to_datetime(df_po['actual_delivery_date'])
df_po['delay_days']             = pd.to_numeric(df_po['delay_days'], errors='coerce')
df_po['po_value_inr']           = pd.to_numeric(df_po['po_value_inr'])

# --- Inventory ---
df_inventory['snapshot_date'] = pd.to_datetime(df_inventory['snapshot_date'])

# --- Sales ---
df_sales['order_date']        = pd.to_datetime(df_sales['order_date'])

print("✅ All data types fixed!")
print(f"\nPurchase Orders date range: {df_po['order_date'].min().date()} to {df_po['order_date'].max().date()}")
print(f"Sales Orders date range:    {df_sales['order_date'].min().date()} to {df_sales['order_date'].max().date()}")
print(f"Inventory date range:       {df_inventory['snapshot_date'].min().date()} to {df_inventory['snapshot_date'].max().date()}")

✅ All data types fixed!

Purchase Orders date range: 2023-01-01 to 2024-12-01
Sales Orders date range:    2023-01-01 to 2024-12-31
Inventory date range:       2023-01-01 to 2024-12-29


In [5]:
# ============================================================
# Business Logic Validation
# These are checks a senior analyst always runs
# ============================================================

print("=" * 55)
print("BUSINESS LOGIC VALIDATION")
print("=" * 55)

# 1. received_qty should never exceed ordered_qty
invalid_receipt = df_po[df_po['received_qty'] > df_po['ordered_qty']]
print(f"\n1. POs where received > ordered:     {len(invalid_receipt)} (should be 0)")

# 2. Defect qty should never exceed received qty
invalid_defect = df_po[df_po['defect_qty'] > df_po['received_qty']]
print(f"2. POs where defects > received:     {len(invalid_defect)} (should be 0)")

# 3. Selling price should always be higher than cost
invalid_margin = df_products[df_products['selling_price_inr'] <= df_products['unit_cost_inr']]
print(f"3. SKUs with negative margin:        {len(invalid_margin)} (should be 0)")

# 4. Cancelled POs should have 0 received qty
cancelled_with_receipt = df_po[(df_po['po_status'] == 'Cancelled') & (df_po['received_qty'] > 0)]
print(f"4. Cancelled POs with received qty:  {len(cancelled_with_receipt)} (should be 0)")

# 5. Actual delivery date should be after order date
df_po_delivered = df_po[df_po['actual_delivery_date'].notna()].copy()
invalid_dates = df_po_delivered[df_po_delivered['actual_delivery_date'] < df_po_delivered['order_date']]
print(f"5. POs delivered before order date:  {len(invalid_dates)} (should be 0)")

# 6. Units fulfilled should never exceed units ordered
invalid_fulfillment = df_sales[df_sales['units_fulfilled'] > df_sales['units_ordered']]
print(f"6. Sales where fulfilled > ordered:  {len(invalid_fulfillment)} (should be 0)")

print(f"\n✅ Business logic validation complete!")

BUSINESS LOGIC VALIDATION

1. POs where received > ordered:     0 (should be 0)
2. POs where defects > received:     0 (should be 0)
3. SKUs with negative margin:        0 (should be 0)
4. Cancelled POs with received qty:  0 (should be 0)
5. POs delivered before order date:  0 (should be 0)
6. Sales where fulfilled > ordered:  0 (should be 0)

✅ Business logic validation complete!


In [6]:
# ============================================================
# Add Derived Columns — These power our analysis later
# ============================================================

# --- Purchase Orders derived columns ---
df_po['fill_rate']        = (df_po['received_qty'] / df_po['ordered_qty']).round(4)
df_po['defect_rate']      = (df_po['defect_qty']   / df_po['received_qty'].replace(0, np.nan)).round(4)
df_po['defect_rate']      = df_po['defect_rate'].fillna(0)
df_po['is_delayed']       = (df_po['delay_days'] > 0).astype(int)
df_po['is_cancelled']     = (df_po['po_status'] == 'Cancelled').astype(int)
df_po['order_month']      = df_po['order_date'].dt.to_period('M').astype(str)
df_po['order_quarter']    = df_po['order_date'].dt.to_period('Q').astype(str)
df_po['order_year']       = df_po['order_date'].dt.year

# --- Sales derived columns ---
df_sales['order_month']   = df_sales['order_date'].dt.to_period('M').astype(str)
df_sales['order_quarter'] = df_sales['order_date'].dt.to_period('Q').astype(str)
df_sales['order_year']    = df_sales['order_date'].dt.year
df_sales['return_value_inr'] = (df_sales['units_returned'] * df_sales['selling_price_inr']).round(2)
df_sales['net_revenue_after_returns'] = (df_sales['net_revenue_inr'] - df_sales['return_value_inr']).round(2)

# --- Inventory derived columns ---
df_inventory['is_low_stock']  = (df_inventory['days_of_inventory'] < 14).astype(int)
df_inventory['is_dead_stock'] = (df_inventory['days_of_inventory'] > 180).astype(int)
df_inventory['stock_month']   = df_inventory['snapshot_date'].dt.to_period('M').astype(str)

# ============================================================
# Save cleaned data to processed folder
# ============================================================

processed_path = "D:/Projects/End-to-end projects/8. Supply Chain Intelligence/Data/Processed"

df_products.to_csv( processed_path + "/products_clean.csv",            index=False)
df_suppliers.to_csv(processed_path + "/suppliers_clean.csv",           index=False)
df_po.to_csv(       processed_path + "/purchase_orders_clean.csv",     index=False)
df_inventory.to_csv(processed_path + "/inventory_snapshots_clean.csv", index=False)
df_sales.to_csv(    processed_path + "/sales_orders_clean.csv",        index=False)

print("✅ All cleaned files saved to processed folder!")
print(f"\nNew columns added:")
print(f"  Purchase Orders: fill_rate, defect_rate, is_delayed, is_cancelled, order_month, order_quarter, order_year")
print(f"  Sales Orders:    order_month, order_quarter, order_year, return_value_inr, net_revenue_after_returns")
print(f"  Inventory:       is_low_stock, is_dead_stock, stock_month")
print(f"\n📁 Saved to: {processed_path}")

✅ All cleaned files saved to processed folder!

New columns added:
  Purchase Orders: fill_rate, defect_rate, is_delayed, is_cancelled, order_month, order_quarter, order_year
  Sales Orders:    order_month, order_quarter, order_year, return_value_inr, net_revenue_after_returns
  Inventory:       is_low_stock, is_dead_stock, stock_month

📁 Saved to: D:/Projects/End-to-end projects/8. Supply Chain Intelligence/Data/Processed
